# Data Preparation

Pipeline unificado de preparação de dados para modelagem:
1. **Segmentação por maturidade** — filtra séries válidas, salva excluídas como backup
2. **Limpeza de outliers** — detecção e tratamento por mediana móvel + MAD
3. **Classificação de demanda** — Croston (Smooth / Erratic / Intermittent / Lumpy)
4. **Geração dos datasets refinados** — 4 parquets por tipo + tags

In [28]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [29]:
# Input
PATH_BASE_DATASET = '../../data/gold/forecasting/datasets/base/ds_sales_timeseries.parquet'
PATH_DIM_TS       = '../../data/gold/data_warehouse/dw_dim_time_series.parquet'

# Output — refined (flat naming)
PATH_REFINED  = '../../data/gold/forecasting/datasets/refined'

# Output — excluídas e tags
PATH_EXCLUDED = '../../data/gold/forecasting/datasets/excluded/ds_sales_timeseries_excluded.parquet'
PATH_TAGS     = '../../data/gold/forecasting/tags/ds_tags.parquet'

os.makedirs(PATH_REFINED, exist_ok=True)
os.makedirs(os.path.dirname(PATH_EXCLUDED), exist_ok=True)
os.makedirs(os.path.dirname(PATH_TAGS), exist_ok=True)

## 1. Segmentação por Maturidade

In [30]:
def segment_by_maturity(df_dim_ts, reference_date=None, discontinued_weeks=26):
    """
    Classifica séries em: valid | cold_start | discontinued

    valid        : histórico suficiente e ativa recentemente
    cold_start   : dados insuficientes para modelagem direta
    discontinued : inativa há mais de `discontinued_weeks` semanas
    """
    df = df_dim_ts.copy()
    df['last_week_date'] = pd.to_datetime(df['last_week_date'])

    if reference_date is None:
        reference_date = df['last_week_date'].max()
    reference_date = pd.to_datetime(reference_date)

    discontinued_threshold = reference_date - pd.Timedelta(weeks=discontinued_weeks)

    df['weeks_since_last_sale'] = ((reference_date - df['last_week_date']).dt.days / 7).astype(int)
    df['segment'] = 'unknown'

    # Descontinuadas (verificar primeiro)
    disc_mask = (
        (df['num_week_with_sales'] == 0) |
        (df['last_week_date'] < discontinued_threshold)
    )
    df.loc[disc_mask, 'segment'] = 'discontinued'

    # Válidas
    valid_mask = (
        (df['segment'] != 'discontinued') &
        (df['total_weeks_length'] >= 4) &
        (df['num_week_with_sales'] >= 2) &
        (df['sales_weeks_ratio'] >= 0.02)
    )
    df.loc[valid_mask, 'segment'] = 'valid'

    # Cold start (restante)
    df.loc[df['segment'] == 'unknown', 'segment'] = 'cold_start'

    counts = df['segment'].value_counts()
    total = len(df)
    print('SEGMENTAÇÃO POR MATURIDADE')
    print('=' * 50)
    print(f'  Referência : {reference_date.date()}')
    print(f'  Threshold  : {discontinued_threshold.date()} (>{discontinued_weeks}w inativa)')
    print()
    for seg, n in counts.items():
        print(f'  {seg:<15}: {n:>5,}  ({n/total*100:.1f}%)')
    print(f'  {"TOTAL":<15}: {total:>5,}')

    assert counts.sum() == total
    return df[['time_series_id', 'segment', 'weeks_since_last_sale']]

In [31]:
df_segments = segment_by_maturity(df_dim_ts)

valid_ids = df_segments.loc[df_segments['segment'] == 'valid', 'time_series_id']
df_valid  = df_base[df_base['time_series_id'].isin(valid_ids)].copy()

print(f'\nDataset válido: {df_valid.shape[0]:,} linhas | {df_valid["time_series_id"].nunique():,} séries')

SEGMENTAÇÃO POR MATURIDADE
  Referência : 2024-10-21
  Threshold  : 2024-04-22 (>26w inativa)

  valid          : 2,232  (78.6%)
  discontinued   :   555  (19.5%)
  cold_start     :    54  (1.9%)
  TOTAL          : 2,841

Dataset válido: 186,052 linhas | 2,232 séries


## 2. Limpeza de Outliers

In [32]:
def detect_outliers_rolling_median(ts_data, window=13, cutoff_multiplier=3.0):
    """Detecção de outliers via mediana móvel + MAD escalado."""
    n = len(ts_data)
    outlier_mask     = np.zeros(n, dtype=bool)
    rolling_baseline = np.full(n, np.nan)
    rolling_mad      = np.full(n, np.nan)

    if (ts_data > 0).sum() < window:
        nan = np.full(n, np.nan)
        return outlier_mask, nan, nan, nan

    for i in range(n):
        start = max(0, i - window // 2)
        end   = min(n, i + window // 2 + 1)
        w = ts_data[start:end]
        w = w[w > 0]
        if len(w) < 3:
            continue
        baseline           = np.median(w)
        rolling_baseline[i] = baseline
        rolling_mad[i]      = np.median(np.abs(w - baseline))

    rolling_std     = 1.4826 * rolling_mad
    upper_threshold = rolling_baseline + cutoff_multiplier * rolling_std
    lower_threshold = np.maximum(rolling_baseline - cutoff_multiplier * rolling_std, 0)

    for i in range(n):
        if ts_data[i] > 0 and not np.isnan(upper_threshold[i]):
            if ts_data[i] > upper_threshold[i] or ts_data[i] < lower_threshold[i]:
                outlier_mask[i] = True

    return outlier_mask, rolling_baseline, upper_threshold, lower_threshold


def run_outlier_pipeline(df, window=13, cutoff_multiplier=3.0):
    """Detecta e trata outliers em todas as séries. Retorna df com colunas de tratamento."""
    result = df.copy()
    result['is_outlier']       = False
    result['rolling_baseline'] = np.nan
    result['upper_threshold']  = np.nan
    result['lower_threshold']  = np.nan

    for ts_id in result['time_series_id'].unique():
        mask   = result['time_series_id'] == ts_id
        ts     = result.loc[mask].sort_values('week_date')
        units  = ts['units_sold'].to_numpy(dtype=float, na_value=0.0)

        outlier_mask, baseline, upper, lower = detect_outliers_rolling_median(
            units, window=window, cutoff_multiplier=cutoff_multiplier
        )
        result.loc[ts.index, 'is_outlier']       = outlier_mask
        result.loc[ts.index, 'rolling_baseline'] = baseline
        result.loc[ts.index, 'upper_threshold']  = upper
        result.loc[ts.index, 'lower_threshold']  = lower

    # Tratamentos
    result['y_capped']       = result['units_sold'].astype('float64')
    result['y_interpolated'] = result['units_sold'].astype('float64')

    capped_mask = result['is_outlier'] & (result['y_capped'] > result['upper_threshold'])
    result.loc[capped_mask, 'y_capped'] = result.loc[capped_mask, 'upper_threshold']

    result.loc[result['is_outlier'], 'y_interpolated'] = np.nan
    result['y_interpolated'] = (
        result.groupby('time_series_id', group_keys=False)['y_interpolated']
        .apply(lambda s: s.interpolate(method='linear', limit_direction='both'))
        .fillna(0)
    )

    total_outliers = result['is_outlier'].sum()
    series_w_outliers = result[result['is_outlier']]['time_series_id'].nunique()
    print('LIMPEZA DE OUTLIERS')
    print('=' * 50)
    print(f'  Janela           : {window} semanas')
    print(f'  Cutoff           : {cutoff_multiplier}×MAD')
    print(f'  Outliers         : {total_outliers:,} ({total_outliers/len(result):.2%} dos registros)')
    print(f'  Séries afetadas  : {series_w_outliers:,}')

    return result

In [33]:
df_treated = run_outlier_pipeline(df_valid, window=13, cutoff_multiplier=3.0)

LIMPEZA DE OUTLIERS
  Janela           : 13 semanas
  Cutoff           : 3.0×MAD
  Outliers         : 12,682 (6.82% dos registros)
  Séries afetadas  : 1,889


## 3. Classificação de Demanda (Croston)

### Plots — Antes vs Depois (Top 20 séries com mais outliers)

In [34]:
import matplotlib.pyplot as plt

PLOT_DIR = '../../plots/outliers/'
os.makedirs(PLOT_DIR, exist_ok=True)


def plot_series_before_after(df_treated, ts_id, save_dir=PLOT_DIR):
    """
    2 painéis para uma série:
      Top    : série original + baseline + thresholds + marcação de outliers
      Bottom : original vs capped vs interpolated
    """
    ts = df_treated[df_treated['time_series_id'] == ts_id].sort_values('week_date')

    dates        = ts['week_date'].values
    original     = ts['units_sold'].astype(float).values
    capped       = ts['y_capped'].values
    interpolated = ts['y_interpolated'].values
    baseline     = ts['rolling_baseline'].values
    upper        = ts['upper_threshold'].values
    lower        = ts['lower_threshold'].values
    is_out       = ts['is_outlier'].values

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Painel 1 — Detecção
    ax1.plot(dates, original, 'o-', color='steelblue', lw=1.5, ms=3, label='Observado', alpha=0.7)
    ax1.plot(dates, baseline, '--', color='green', lw=2, label='Baseline (mediana móvel)', alpha=0.8)
    ax1.plot(dates, upper, ':', color='red', lw=1.5, label='Upper threshold', alpha=0.7)
    ax1.plot(dates, lower, ':', color='orange', lw=1.5, label='Lower threshold', alpha=0.7)
    ax1.fill_between(dates, lower, upper, color='green', alpha=0.08)
    if is_out.sum() > 0:
        ax1.scatter(dates[is_out], original[is_out], color='red', s=120, marker='X', zorder=5,
                    label=f'Outliers ({is_out.sum()})')
    ax1.set_title(f'Série {ts_id} — Detecção', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Unidades vendidas')
    ax1.legend(fontsize=9, loc='upper left')
    ax1.grid(alpha=0.3)

    # Painel 2 — Comparação
    ax2.plot(dates, original, 'o-', color='steelblue', lw=1.5, ms=3, label='Original', alpha=0.4)
    ax2.plot(dates, capped, 's-', color='darkorange', lw=1.5, ms=3, label='Capped', alpha=0.8)
    ax2.plot(dates, interpolated, 'D-', color='purple', lw=1.5, ms=3, label='Interpolated', alpha=0.8)
    if is_out.sum() > 0:
        ax2.scatter(dates[is_out], original[is_out], color='red', s=60, zorder=5,
                    label=f'Outliers originais ({is_out.sum()})')
    ax2.set_title(f'Série {ts_id} — Antes vs Depois', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Data')
    ax2.set_ylabel('Unidades vendidas')
    ax2.legend(fontsize=9, loc='upper left')
    ax2.grid(alpha=0.3)
    ax2.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    path = os.path.join(save_dir, f'serie_{ts_id}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    return path


def generate_before_after_plots(df_treated, n=20, save_dir=PLOT_DIR):
    """Gera plots das N séries com mais outliers."""
    top_series = (
        df_treated[df_treated['is_outlier']]
        .groupby('time_series_id').size()
        .nlargest(n).index.tolist()
    )
    print(f'Gerando {len(top_series)} plots em {save_dir!r} ...')
    for i, ts_id in enumerate(top_series, 1):
        path = plot_series_before_after(df_treated, ts_id, save_dir)
        print(f'  [{i:>2}/{len(top_series)}] {path}')
    print(f'\nPlots salvos em: {save_dir}')


generate_before_after_plots(df_treated, n=20)

Gerando 20 plots em '../../plots/outliers/' ...
  [ 1/20] ../../plots/outliers/serie_1041.png
  [ 2/20] ../../plots/outliers/serie_1347.png
  [ 3/20] ../../plots/outliers/serie_755.png
  [ 4/20] ../../plots/outliers/serie_388.png
  [ 5/20] ../../plots/outliers/serie_575.png
  [ 6/20] ../../plots/outliers/serie_1277.png
  [ 7/20] ../../plots/outliers/serie_386.png
  [ 8/20] ../../plots/outliers/serie_455.png
  [ 9/20] ../../plots/outliers/serie_597.png
  [10/20] ../../plots/outliers/serie_1496.png
  [11/20] ../../plots/outliers/serie_1630.png
  [12/20] ../../plots/outliers/serie_2225.png
  [13/20] ../../plots/outliers/serie_691.png
  [14/20] ../../plots/outliers/serie_1011.png
  [15/20] ../../plots/outliers/serie_1200.png
  [16/20] ../../plots/outliers/serie_723.png
  [17/20] ../../plots/outliers/serie_1070.png
  [18/20] ../../plots/outliers/serie_1097.png
  [19/20] ../../plots/outliers/serie_1210.png
  [20/20] ../../plots/outliers/serie_2264.png

Plots salvos em: ../../plots/outliers/


In [35]:
def classify_series(y: pd.Series) -> str:
    """
    Croston demand classification via ADI × CV².
    Thresholds: Syntetos et al. (2005) — ADI=1.32, CV²=0.49
    ddof=0 para evitar NaN em séries com única observação não-zero.
    """
    non_zero = y[y > 0]
    if len(non_zero) == 0:
        return 'lumpy'
    adi = len(y) / len(non_zero)
    cv2 = (non_zero.std(ddof=0) / non_zero.mean()) ** 2 if non_zero.mean() != 0 else 0
    if adi < 1.32:
        return 'smooth' if cv2 < 0.49 else 'erratic'
    else:
        return 'intermittent' if cv2 < 0.49 else 'lumpy'


demand_type_map = (
    df_treated
    .groupby('time_series_id')['y_capped']
    .apply(classify_series)
)
df_treated['demand_type'] = df_treated['time_series_id'].map(demand_type_map)

print('CLASSIFICAÇÃO DE DEMANDA')
print('=' * 50)
counts = df_treated[['time_series_id','demand_type']].drop_duplicates()['demand_type'].value_counts()
for dt, n in counts.items():
    print(f'  {dt:<15}: {n:>5,} séries')

CLASSIFICAÇÃO DE DEMANDA
  intermittent   :   700 séries
  lumpy          :   557 séries
  smooth         :   542 séries
  erratic        :   433 séries


## 4. Salvar Outputs

In [36]:
# -------------------------------------------------------------------------
# 4.1 Refined datasets — formato padrão time series: unique_id | ds | y
# -------------------------------------------------------------------------

DEMAND_TYPES = ['smooth', 'erratic', 'intermittent', 'lumpy']

for demand_type in DEMAND_TYPES:
    df_out = (
        df_treated[df_treated['demand_type'] == demand_type]
        [['time_series_id', 'week_date', 'y_capped', 'demand_type']]
        .rename(columns={'time_series_id': 'unique_id', 'week_date': 'ds', 'y_capped': 'y'})
        .sort_values(['unique_id', 'ds'])
        .reset_index(drop=True)
    )
    path = f'{PATH_REFINED}/ds_sales_timeseries_{demand_type}.parquet'
    df_out.to_parquet(path, index=False)
    print(f'  Salvo {demand_type:<15}: {df_out["unique_id"].nunique():>5,} séries | {path}')

  Salvo smooth         :   542 séries | ../../data/gold/forecasting/datasets/refined/ds_sales_timeseries_smooth.parquet
  Salvo erratic        :   433 séries | ../../data/gold/forecasting/datasets/refined/ds_sales_timeseries_erratic.parquet
  Salvo intermittent   :   700 séries | ../../data/gold/forecasting/datasets/refined/ds_sales_timeseries_intermittent.parquet
  Salvo lumpy          :   557 séries | ../../data/gold/forecasting/datasets/refined/ds_sales_timeseries_lumpy.parquet


In [37]:
# -------------------------------------------------------------------------
# 4.2 Backup das séries excluídas
# -------------------------------------------------------------------------

excluded_ids = df_segments[df_segments['segment'] != 'valid'][['time_series_id', 'segment', 'weeks_since_last_sale']]

df_excluded = (
    df_dim_ts[['time_series_id', 'first_week_date', 'last_week_date',
               'total_weeks_length', 'num_week_with_sales', 'sales_weeks_ratio']]
    .merge(excluded_ids, on='time_series_id')
    .assign(reason=lambda d: d['segment'].map({
        'discontinued': 'inactive >26 weeks or never sold',
        'cold_start':   'insufficient history or single sale'
    }))
)

df_excluded.to_parquet(PATH_EXCLUDED, index=False)
print(f'  Excluídas salvas: {len(df_excluded):,} séries | {PATH_EXCLUDED}')
df_excluded['segment'].value_counts()

  Excluídas salvas: 609 séries | ../../data/gold/forecasting/datasets/excluded/ds_sales_timeseries_excluded.parquet


segment
discontinued    555
cold_start       54
Name: count, dtype: int64

In [38]:
# -------------------------------------------------------------------------
# 4.3 Tags — metadados por série (todas as 2,841)
# -------------------------------------------------------------------------

SUGGESTED_MODEL = {
    'discontinued': 'end_cycle',
    'cold_start':   'similarity',
    'smooth':       'global',
    'erratic':      'statistical',
    'intermittent': 'statistical',
    'lumpy':        'statistical',
}

valid_demand = df_treated[['time_series_id', 'demand_type']].drop_duplicates()

df_tags = (
    df_segments[['time_series_id', 'segment']]
    .merge(valid_demand, on='time_series_id', how='left')
    .assign(suggested_model=lambda d: d.apply(
        lambda r: SUGGESTED_MODEL.get(r['demand_type'], SUGGESTED_MODEL.get(r['segment'])), axis=1
    ))
)

df_tags.to_parquet(PATH_TAGS, index=False)
print(f'  Tags salvas: {len(df_tags):,} séries | {PATH_TAGS}')
df_tags['suggested_model'].value_counts()

  Tags salvas: 2,841 séries | ../../data/gold/forecasting/tags/ds_tags.parquet


suggested_model
statistical    1690
end_cycle       555
global          542
similarity       54
Name: count, dtype: int64

## Resumo

In [39]:
print('OUTPUTS GERADOS')
print('=' * 70)
print()
print('  datasets/refined/')
for dt in DEMAND_TYPES:
    n = df_treated[df_treated['demand_type'] == dt]['time_series_id'].nunique()
    print(f'    {dt}/ds_sales_timeseries.parquet   → {n:,} séries')
print()
print(f'  datasets/excluded/ds_sales_timeseries_excluded.parquet')
print(f'    discontinued : {(df_excluded["segment"]=="discontinued").sum():,}')
print(f'    cold_start   : {(df_excluded["segment"]=="cold_start").sum():,}')
print()
print(f'  tags/ds_tags.parquet   → {len(df_tags):,} séries')
print()
print('  Colunas dos refined datasets: unique_id | ds | y | demand_type')

OUTPUTS GERADOS

  datasets/refined/
    smooth/ds_sales_timeseries.parquet   → 542 séries
    erratic/ds_sales_timeseries.parquet   → 433 séries
    intermittent/ds_sales_timeseries.parquet   → 700 séries
    lumpy/ds_sales_timeseries.parquet   → 557 séries

  datasets/excluded/ds_sales_timeseries_excluded.parquet
    discontinued : 555
    cold_start   : 54

  tags/ds_tags.parquet   → 2,841 séries

  Colunas dos refined datasets: unique_id | ds | y | demand_type
